In [1]:
import os
from re import T

import numpy as np
from scipy.optimize import curve_fit
import pandas as pd
import math
import time 


from utils.config import load_config, Configuration
from utils.utils import retrieve_stacked_betas, retrieve_roi_mask, filter_roi_mask, retrieve_stacked_betas_test
import glob

import logging


logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)

In [2]:
def gaussian_2d_curve(independent, x0, y0, sigma, slope, intercept):
    X, Y = independent
    x = X - x0
    y = Y - y0
    return (np.exp(-0.5 * ((x/sigma)**2 + (y/sigma)**2)) * slope + intercept)

def gaussian_2d_curve_pol(independent, ecc, pol, sigma, slope, intercept):  
    X,Y=independent
    x0,y0 = pol2cart(ecc,pol)
    x = X - x0
    y = Y - y0
    return (np.exp(-0.5 * ((x/sigma)**2 + (y/sigma)**2)) * slope + intercept)

def pol2cart(rho, phi):
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return(x, y)

In [3]:
params = {'random': True, 
          'initial': (np.array([-0.4, -0.4, 0.01, 0.1, - 2]), np.array([0.4, 0.4, 2, 10, 2])), 
           'bounds': (np.array([-1.05, -1.05, 0.01, 1e-08, - np.inf]),np.array([1.05, 1.05, np.inf, np.inf, np.inf])),
           'loss': 'linear',
           'method': 'trf'
        }




subj_list = [1]
mode = "averaged"

In [10]:
config = load_config("config.yaml")
rois = config.rois_to_analyze


mask = retrieve_roi_mask(config, 1)
indices = filter_roi_mask(20, mask)[0]

for index in indices:
    print(index)
    break

2025-01-28 14:36:21 INFO: Loading mask from
data/t_test_roi/face_animate/lh.subj01.testrois.mgz
data/t_test_roi/face_animate/rh.subj01.testrois.mgz


25950


In [ ]:
from tqdm import tqdm
# mds_file = "data/mds_dir/subj_01/1_mask20_mds_sample0.npy"
# mds_file = os.path.join("data/mds_dir", f"subj_{subj:02d}", f"{subj}_mask20_mds_sample0.npy")


# TODO
# - load mask
# for each ROI
#   - get MDS space for corresponding ROI
#   - get voxels for corresponding ROI
#   - fit gaussian for each voxel of ROI
#   - save fit

targetspace = 'nativesurface'
columns = ["x0", "y0", "sigma", "slope", "intercept"]
initial = params['initial']
bounds = params['bounds']


for i, subj in enumerate(subj_list):    
    gaussian_fit_result_dir_path = os.path.join(config.gaussian_fit_results_dir, f"subj{subj:02d}")
    os.makedirs(gaussian_fit_result_dir_path, exist_ok= True)
    
    mask = retrieve_roi_mask(config, subj)
    betas, _ = retrieve_stacked_betas(config, subj, "averaged", 0)
    betas_test = retrieve_stacked_betas_test(config, subj)

    if np.isnan(betas).any():
        raise ValueError("Found NaNs")
        print(f'FOUND NANS in subj0{i+1} s betas')

    # if mode == "train":
    #     betas_test_file = os.path.join(betas_dir, f'{subj}_betas_list_{targetspace}_test.npy')
    #     print(f'LOADING TEST BETAS FOR SUBJ0{i+1}')
    #     betas_test = np.load(betas_test_file, allow_pickle=False).astype(np.float32).T

    #     train_test_mask_file = os.path.join(betas_dir, f'{subj}_betas_list_{targetspace}_train_test_mask.npy')
    #     print(f'LOADING TRAIN TEST MSAK FOR SUBJ0{i+1}')
    #     train_test_mask = np.load(train_test_mask_file, allow_pickle=False).T.astype(bool)


    logging.info(f'Starting fitting for subj{subj:02d}')
    n_betas, n_voxels = betas.shape


    model_allROI = np.zeros((len(rois), n_voxels, len(columns)))

    for i, roi in enumerate(list(rois.keys())):
        logging.info(f'Fitting for ROI {roi}')
        roi_mask_value = rois[roi]

        gaussian_result_file_path = os.path.join(gaussian_fit_result_dir_path, f"fitted_voxels_mask_{roi_mask_value}.xlsx") 

        if os.path.exists(gaussian_result_file_path):
            logging.info("Loading existing gaussian fit results")
            fits_roi = pd.read_excel(gaussian_result_file_path)


        else:
            mds_file = os.path.join("data/mds_dir", f"subj_{subj:02d}", f"mask_{roi_mask_value}_averaged_mds.npy")

            # mds_file = os.path.join(mds_dir, subj, f'{subj}_{roi}_MDS_rotated_VO-1_{mode}.npy')  # we only care about rotate
            mds = np.load(mds_file, allow_pickle=True).astype(np.float32).T

                
            

            mask_voxel_indices = filter_roi_mask(roi_mask_value, mask)
            mask_voxel_indices = mask_voxel_indices[0]
            fits_roi = pd.DataFrame(columns=columns)

            for voxel_i in tqdm(mask_voxel_indices):
                voxel_activity = betas[:, voxel_i]

                if params['random']:
                    attempt = 1
                    solved = False
                    while not solved and attempt <= 10:
                        try:  
                            initial_guess = (initial[1] - initial[0]) * np.random.random(initial[0].shape) + initial[0]
                            voxel_fit = curve_fit(
                                    gaussian_2d_curve,
                                    mds,
                                    voxel_activity,
                                    p0 = initial_guess,
                                    bounds = bounds,
                                    method = 'trf',
                                    ftol = 1e-06,
                            )[0]
                            solved = True
                        
                        except RuntimeError:
                            if attempt >= 5:
                                print(f'VOXEL {voxel_i}: optimal params not found after {attempt} attempts')
                            
                            attempt + 1

                fits_roi.loc[voxel_i] = voxel_fit
            

            fits_roi.to_excel(gaussian_result_file_path)

        
        # calculate stats
        epsilon_stabilizer = 0 # 1e-8
        masked_betas = ((betas.T)[mask_voxel_indices]).T

        def gaus_roi(fits):
            return gaussian_2d_curve(mds, *fits)

        pred_activity = fits_roi.apply(gaus_roi, axis=1)
        pred_activity = np.array([np.array(x) for x in pred_activity]).T

        roi_res = np.sum((pred_activity - masked_betas)**2, axis=0)
        roi_tot = np.sum((masked_betas - np.tile(masked_betas.mean(axis=0), (masked_betas.shape[0], 1)))**2, axis=0)

        fits_roi["var_explained"] = 1 - roi_res / (roi_tot + epsilon_stabilizer)



#     for voxel in tqdm(range(n_voxels)):
#         voxel_activity = betas[:, voxel]

#         for i, roi in enumerate(list(rois.keys())):
#             mds_file = os.path.join("data/mds_dir", f"subj_{subj:02d}", f"{subj}_mask20_mds_sample0.npy")

#             # mds_file = os.path.join(mds_dir, subj, f'{subj}_{roi}_MDS_rotated_VO-1_{mode}.npy')  # we only care about rotate
#             mds = np.load(mds_file, allow_pickle=True).astype(np.float32).T

#             fit_file = os.path.join("data", f"fits_{subj}_{mode}_{roi}_inversed.npy")
                
#             if os.path.exists(fit_file):
#                 print(f'a fitted model already exists for {roi}, already exist \n Delete it if refitting \
#                         or fitting new voxels is needed ')
#                 continue   # This is not ideal. But I dont want to create a model for each voxel 

            
#             if params['random']:
#                 attempt = 1
#                 solved = False
#                 while not solved and attempt <= 10:
#                     try:  
#                         initial_guess = (initial[1] - initial[0]) * np.random.random(initial[0].shape) + initial[0]
#                         voxel_fit = curve_fit(
#                                 gaussian_2d_curve_pol,
#                                 mds,
#                                 voxel_activity,
#                                 p0 = initial_guess,
#                                 bounds = bounds,
#                                 method = 'trf',
#                                 ftol = 1e-06,
#                         )[0]
#                         solved = True
                    
#                     except RuntimeError:
#                         print(f'VOXEL {voxel}: optimal params not found after {attempt} attempts')
#                         attempt + 1
        
#             model_allROI[i, voxel] = voxel_fit # put the fitted values on the rigth voxel to ROI index 
#         if not voxel % 100:
#             print(f'\t\tFitted Voxel {voxel} out of {n_voxels}, elapsed time on {subj}: {time.strftime("%H:%M:%S", time.gmtime(time.time() - start))}' )
    
#     for i, roi in enumerate(list(rois.keys())):
#         fit_file = os.path.join("data", f"fits_{subj}_{mode}_{roi}_inversed.npy")
                
#         if os.path.exists(fit_file):
#             print(f'a fitted model already exists for {roi}, skipping')
#             continue
            
#         fits_roi = pd.DataFrame(model_allROI[i], columns=columns)

#         mds_file = os.path.join("data/mds_dir", f"subj_{subj:02d}", f"{subj}_mask20_mds_sample0.npy")  # Works better if I dont forget this  
#         mds = np.load(mds_file, allow_pickle=True).astype(np.float32).T
        
#         def gaus_roi(fits):
#             return gaussian_2d_curve_pol(mds, *fits)

#         pred_activity = fits_roi.apply(gaus_roi, axis = 1)
#         pred_activity = np.array([np.array(x) for x in pred_activity]).T
#         roi_res = np.sum((pred_activity - betas)**2, axis=0)
#         roi_tot = sum((betas- np.tile(betas.mean(axis=0), (n_betas, 1)))**2).T

#         # At that point mode will always be train 
#         if mode == "train":
#             mds_test = mds[:, train_test_mask]  # mds_test is a misleading name: the mds is not create from the test data, but the train data
#             def gaus_roi_test(fits):
#                 return gaussian_2d_curve_pol(mds_test, *fits)
#             pred_activity_test = fits_roi.apply(gaus_roi_test, axis=1)
#             pred_activity_test = np.array([np.array(x) for x in pred_activity_test]).T
#             roi_res_test = np.sum((pred_activity_test - betas_test)**2, axis=0)
#             roi_rot_test = sum((betas_test - np.tile(betas_test.mean(axis=0), (sum(train_test_mask), 1)))**2).T
#             fits_roi["test_var_explained"] = 1 - roi_res_test / roi_rot_test
#         fits_roi["var_explained"] = 1 - roi_res / roi_tot
#         fits_roi["mds_ecc"] = (fits_roi.x0 ** 2 + fits_roi.y0[1] ** 2) ** (1/2)
#         fits_roi["mds_ang"] = np.arctan2(fits_roi.x0/bounds[1][0], fits_roi.y0/bounds[1][1])

#         np.save(fit_file, fits_roi)
#         print(f'file for {roi} has been saved ')
# del betas

#gaussian_fit(subj_list, rois, params,  mode="train")


NameError: name 'fits_roi' is not defined